# Quick start

This notebook loads a small radial feeder written in `.cimtbl`, looks at what the loader made, and saves it as CIM RDF/XML. It runs from `docs/guide/` in the repository, next to [`tiny_feeder.cimtbl`](https://github.com/PNNL-CIM-Tools/CIM-Tables/blob/develop/docs/guide/tiny_feeder.cimtbl).

## The file

One source, two line segments and two loads. Every table is a CIM class, every column a CIM attribute or association, and the units are on the headers.

In [1]:
from pathlib import Path

print(Path('tiny_feeder.cimtbl').read_text())

Profile = cim18v17
Authority = https://example.com/models
Model = tiny_feeder

Object BasePower: mRID=sbase, basePower (MVA)=100
Object Substation: mRID=sub
Object Feeder: mRID=fdr, NormalEnergizingSubstation=sub

Table BaseVoltage: mRID, nominalVoltage (kV)
    bv_12kv, 12.47

Object EnergySource: mRID=source, node=sourcebus, phases=ABC, nominalVoltage (kV)=12.47, BaseVoltage=bv_12kv, EquipmentContainer=fdr

Table ACLineSegment: mRID, node1, node2, phases, length (ft), r (ohm), x (ohm), BaseVoltage, EquipmentContainer=fdr
    ln_trunk,  sourcebus, n1, ABC, 2000, 0.38, 0.75, bv_12kv
    ln_branch, n1,        n2, C,   500,  0.12, 0.20, bv_12kv

Table EnergyConsumer: mRID, node, phases, p (kW), q (kVAr), BaseVoltage, EquipmentContainer=fdr
    ld_n1, n1, ABC, 1155, 660, bv_12kv
    ld_n2, n2, C,   170,  80,  bv_12kv



## Load it

`load_model` builds a new in-memory cim-graph model. The file has one `Feeder`, so the result is a `FeederModel` with that Feeder as its container.

In [2]:
import cimtables

network = cimtables.load_model('tiny_feeder.cimtbl')
type(network).__name__, network.container.mRID

('FeederModel', 'fdr')

## What the loader made

The file declares 9 objects. The model holds more: CIM puts equipment on `Terminal`s, `Terminal`s on `ConnectivityNode`s, and per-phase data in child objects, and the `node` and `phases` columns generated all of them.

In [3]:
for cim_class, objects in network.graph.items():
    print(f'{cim_class.__name__:20} {len(objects):2}  ' + ', '.join(sorted(o.mRID for o in objects.values())))

BasePower             1  sbase
Substation            1  sub
Feeder                1  fdr
BaseVoltage           1  bv_12kv
ConnectivityNode      3  n1, n2, sourcebus
Terminal              7  ld_n1_T1, ld_n2_T1, ln_branch_T1, ln_branch_T2, ln_trunk_T1, ln_trunk_T2, source_T1
EnergySourcePhase     3  source_A, source_B, source_C
EnergySource          1  source
ACLineSegmentPhase    4  ln_branch_C, ln_trunk_A, ln_trunk_B, ln_trunk_C
ACLineSegment         2  ln_branch, ln_trunk
EnergyConsumerPhase   4  ld_n1_A, ld_n1_B, ld_n1_C, ld_n2_C
EnergyConsumer        2  ld_n1, ld_n2


Generated objects are named after their owner: its mRID plus `_T1`, `_T2` for Terminals and plus the phase for per-phase children (`ln_trunk_T1`, `ln_trunk_A`). Each line's Terminals sit on the nodes named in its `node1` and `node2` cells:

In [4]:
trunk = network.find_by_mRID('ln_trunk')
for terminal in trunk.Terminals:
    print(terminal.mRID, terminal.sequenceNumber, terminal.ConnectivityNode.mRID, terminal.phases.value)

ln_trunk_T1 1 sourcebus ABC
ln_trunk_T2 2 n1 ABC


In [5]:
branch = network.find_by_mRID('ln_branch')
[phase.mRID for phase in branch.ACLineSegmentPhases], branch.Terminals[0].phases.value

(['ln_branch_C'], 'C')

## Units

The file says feet and kilowatts; the model holds SI, as CIM does. Convert back with `.to()`.

In [6]:
load = network.find_by_mRID('ld_n1')
print(f'length {float(trunk.length):.1f} m = {trunk.length.to("ft"):.0f} ft')
print(f'p      {float(load.p):.0f} W = {load.p.to("kW"):.0f} kW')

length 609.6 m = 2000 ft
p      1155000 W = 1155 kW


## Identity

Each UUID is derived from the object's mRID and the file's `Authority` and `Model`, so loading the file again gives the same UUIDs.

In [7]:
again = cimtables.load_model('tiny_feeder.cimtbl')
trunk.identifier == again.find_by_mRID('ln_trunk').identifier

True

## Save it as CIM

The model is ordinary cim-graph, so cim-graph's writers save it.

In [8]:
import tempfile
from cimgraph.utils import write_xml

xml_path = Path(tempfile.mkdtemp()) / 'tiny_feeder.xml'
write_xml(network, str(xml_path))

lines = xml_path.read_text().splitlines()
start = next(i for i, line in enumerate(lines) if '<cim:EnergyConsumer ' in line)
print('\n'.join(lines[start:start + 12]))

<cim:EnergyConsumer rdf:about="urn:uuid:7290a692-f3a3-591c-b4c9-4651735594f7">
  <cim:Identity.identifier>7290a692-f3a3-591c-b4c9-4651735594f7</cim:Identity.identifier>
  <cim:EnergyConsumer.p>170000.0</cim:EnergyConsumer.p>
  <cim:EnergyConsumer.q>80000.0</cim:EnergyConsumer.q>
  <cim:ConductingEquipment.BaseVoltage rdf:resource="urn:uuid:de982f74-cd7f-5271-8c1c-39d8281c221a"/>
  <cim:Equipment.EquipmentContainer rdf:resource="urn:uuid:8f6a62b8-6f6d-5040-add2-7f2ef1b202cd"/>
  <cim:IdentifiedObject.mRID>ld_n2</cim:IdentifiedObject.mRID>
</cim:EnergyConsumer>
<cim:EnergyConsumer rdf:about="urn:uuid:c20cd919-9ba6-572a-969f-ae704331f970">
  <cim:Identity.identifier>c20cd919-9ba6-572a-969f-ae704331f970</cim:Identity.identifier>
  <cim:EnergyConsumer.p>1155000.0</cim:EnergyConsumer.p>
  <cim:EnergyConsumer.q>660000.0</cim:EnergyConsumer.q>


## Next

- [The format](format.md) covers every statement and cell rule.
- [Connectivity and phases](connectivity.md) covers the shorthand columns.
- [CIM XML to .cimtbl](round_trip.ipynb) goes the other way.